In [1]:
import os
import tqdm.auto
from tqdm.std import tqdm as std_tqdm
tqdm.auto.tqdm = std_tqdm
from forget.model.base import AutoModelForCausalLMWrapper

# llama3

```bash
chat format:
    <|begin_of_text|>
    
    <|start_header_id|>system<|end_header_id|>
    system text<|eot_id|>

    <|start_header_id|>user<|end_header_id|>
    user text<|eot_id|>
    
    <|start_header_id|>assistant<|end_header_id|>
    assistant text<|eot_id|>
```

In [ ]:
# Chat formatting tags
B_SYS = "<|start_header_id|>system<|end_header_id|>"
E_SYS = "<|eot_id|>"
B_USER = "<|start_header_id|>user<|end_header_id|>"
E_USER = "<|eot_id|>"
B_ASSISTANT = "<|start_header_id|>assistant<|end_header_id|>"
E_ASSISTANT = "<|eot_id|>"

# Position markers for activation steering
# Steer from the assistant header onwards
ADD_FROM_POS_CHAT = B_ASSISTANT

# End of prompt sequence token tensor
END_STR = None

# Special tokens
BOS = "<|begin_of_text|>"
EOS = "<|eot_id|>"

In [4]:
model_path = "meta-llama/Llama-3.1-8B-Instruct"
llm = AutoModelForCausalLMWrapper(
    hf_token=os.getenv("HF_TOKEN"),
    model_path=model_path,
    tokenizer_path=model_path,
    instruction_end_marker=ADD_FROM_POS_CHAT,
)

Loading weights: 100%|██████████| 291/291 [00:00<00:00, 14722.71it/s]


In [ ]:
prompt = f"{BOS}{B_SYS}You are a helpful assistant.{E_SYS}{B_USER}What is the capital of France?{E_USER}{B_ASSISTANT}"
generated = llm.batch_generate([prompt], max_new_tokens=64)[0]
generated.split(B_ASSISTANT, maxsplit=1)[-1]

'\n\nThe capital of France is Paris.<|eot_id|>'

# Qwen

```bash
chat format:
    <|im_start|>system
    system text<|im_end|>
    <|im_start|>user
    user text<|im_end|>
    <|im_start|>assistant
    assistant text<|im_end|>
```

In [2]:
# Chat formatting tags
B_SYS = "<|im_start|>system\n"
E_SYS = "<|im_end|>\n"
B_USER = "<|im_start|>user\n"
E_USER = "<|im_end|>\n"
B_ASSISTANT = "<|im_start|>assistant\n"
E_ASSISTANT = "<|im_end|>\n"

# Position markers for activation steering
ADD_FROM_POS_CHAT = B_ASSISTANT

END_STR = None

# Special tokens
BOS = ""  # Qwen2.5 doesn't prepend a BOS in its chat template
EOS = "<|im_end|>"

prompt = f"{BOS}{B_SYS}You are a helpful assistant.{E_SYS}{B_USER}What is the capital of France?{E_USER}{B_ASSISTANT}"

In [3]:
model_path = "Qwen/Qwen2.5-7B-Instruct"
llm = AutoModelForCausalLMWrapper(
    hf_token=os.getenv("HF_TOKEN"),
    model_path=model_path,
    tokenizer_path=model_path,
    instruction_end_marker=ADD_FROM_POS_CHAT,
)

Loading weights: 100%|██████████| 339/339 [00:00<00:00, 14584.47it/s]


In [4]:
prompt = f"{BOS}{B_SYS}You are a helpful assistant.{E_SYS}{B_USER}What is the capital of France?{E_USER}{B_ASSISTANT}"
generated = llm.batch_generate([prompt], max_new_tokens=64)[0]
generated.split(B_ASSISTANT, maxsplit=1)[-1]

'The capital of France is Paris.<|im_end|>'

In [14]:
llm.layer_wrappers[0].activations.shape

torch.Size([1, 1, 3584])